In [2]:
from pathlib import Path

print(Path.cwd())

c:\Users\shafi\number-simplex-reproduction\notebooks1


In [4]:
import sys
from pathlib import Path

cwd = Path.cwd()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise RuntimeError(
        "Could not find the src folder. "
        f"Current directory is: {cwd}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: c:\Users\shafi\number-simplex-reproduction
src exists: True


In [5]:
from src.firing_rate_tuning import analyze_neuron
from src.number_decoding_analysis import make_temporal_features
from src.poisson_naive_bayes import cv_poisson_nb

In [6]:
result = analyze_neuron(
    "YFU",
    43,
    root=PROJECT_ROOT,
    plot=False,
    verbose=True,
)

Subject: YFU

Total recorded neurons: 78

All recorded regions:
  acc: 16
  amy: 23
  hpc: 39

MTL regions used in paper:
  amy: 23
  hpc: 39

Total MTL neurons: 62

Selected neuron:
  MATLAB row: 43
  Python row: 42
  Region: hpc
  Total spikes: 15270

Number presentation counts:
number
1    53
2    67
3    52
4    62
5    51
6    48
7    65
8    49
9    47
Name: count, dtype: int64

Firing-rate tuning:
        count      mean       std       sem
number                                     
1          53  6.498952  2.725109  0.374323
2          67  6.766169  2.706018  0.330593
3          52  7.307692  3.003638  0.416530
4          62  6.308244  2.836531  0.360240
5          51  7.189542  2.857547  0.400137
6          48  7.129630  2.720046  0.392605
7          65  6.034188  2.484281  0.308137
8          49  6.190476  2.585970  0.369424
9          47  7.092199  2.447003  0.356932


In [8]:
presentations = result["presentations"]
spike_times = result["spike_times"]

print("Presentations:", len(presentations))
print(presentations.head())
print("Number of spikes:", len(spike_times))

Presentations: 494
   trial  number          onset  operand  spike_count  firing_rate
0      1       7    6660.833333        1            6     6.666667
1      2       7   81376.233333        1            6     6.666667
2      3       7   89626.100000        1            6     6.666667
4      5       8  118708.933333        1            4     4.444444
5      6       4  124642.166667        1            3     3.333333
Number of spikes: 15270


In [9]:
X, y, edges = make_temporal_features(
    presentations,
    spike_times,
    bin_size=100,
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Bin edges:", edges)

print("\nMinimum value in X:", X.min())
print("Maximum value in X:", X.max())

print("\nFirst 5 rows:")
print(X[:5])

print("\nFirst 20 labels:")
print(y[:20])

X shape: (494, 9)
y shape: (494,)
Bin edges: [ 50. 150. 250. 350. 450. 550. 650. 750. 850. 950.]

Minimum value in X: 0.0
Maximum value in X: 6.0

First 5 rows:
[[1. 0. 1. 0. 0. 0. 0. 2. 2.]
 [1. 0. 1. 1. 1. 1. 0. 1. 0.]
 [3. 0. 0. 0. 0. 1. 2. 0. 0.]
 [0. 0. 1. 0. 0. 1. 1. 0. 1.]
 [0. 1. 0. 0. 1. 0. 0. 1. 0.]]

First 20 labels:
[7 7 7 8 4 7 3 4 6 7 7 9 7 1 3 3 7 4 5 6]


In [10]:
import numpy as np

classes, counts = np.unique(y, return_counts=True)

print("\nClass counts:")
for c, n in zip(classes, counts):
    print(f"Number {c}: {n}")


Class counts:
Number 1: 53
Number 2: 67
Number 3: 52
Number 4: 62
Number 5: 51
Number 6: 48
Number 7: 65
Number 8: 49
Number 9: 47


In [11]:
poisson_result = cv_poisson_nb(
    X,
    y,
    n_splits=5,
    grouped=False,
)

print(
    "Poisson Naive Bayes accuracy:",
    poisson_result["accuracy"]
)

print(
    "Chance level:",
    1 / 9
)

Poisson Naive Bayes accuracy: 0.1214574898785425
Chance level: 0.1111111111111111


In [12]:
print("Posterior shape:", poisson_result["posterior"].shape)

print("\nFirst posterior:")
print(poisson_result["posterior"][0])

print("\nTrue number:")
print(poisson_result["y_true"][0])

print("\nPredicted number:")
print(poisson_result["y_pred"][0])

print("\nPosterior sum:")
print(poisson_result["posterior"][0].sum())

Posterior shape: (494, 9)

First posterior:
[0.04794795 0.05950325 0.1752669  0.07907041 0.02028963 0.04386829
 0.08277947 0.09650266 0.39477143]

True number:
7

Predicted number:
9

Posterior sum:
0.9999999999999993


In [13]:
groups = presentations["trial"].to_numpy()

poisson_grouped = cv_poisson_nb(
    X,
    y,
    groups=groups,
    n_splits=5,
    grouped=True,
)

print(
    "Grouped Poisson NB accuracy:",
    poisson_grouped["accuracy"]
)

print(
    "Chance:",
    1 / 9
)

Grouped Poisson NB accuracy: 0.11538461538461539
Chance: 0.1111111111111111


In [14]:
import pandas as pd
import numpy as np

posterior = poisson_grouped["posterior"]
y_true = poisson_grouped["y_true"]

posterior_matrix = np.zeros((9, 9))

for true_number in range(1, 10):
    mask = y_true == true_number

    posterior_matrix[true_number - 1] = (
        posterior[mask].mean(axis=0)
    )

posterior_df = pd.DataFrame(
    posterior_matrix,
    index=[f"True {k}" for k in range(1, 10)],
    columns=[f"P({k})" for k in range(1, 10)],
)

posterior_df

,P(1),P(2),P(3),P(4),P(5),P(6),P(7),P(8),P(9)
True 1,0.115558,0.113492,0.106159,0.105784,0.123945,0.108111,0.108232,0.106638,0.112082
True 2,0.109089,0.106271,0.110458,0.113423,0.117860,0.111535,0.111743,0.120798,0.098824
True 3,0.103527,0.112814,0.122007,0.109848,0.119479,0.104015,0.104485,0.105214,0.118611
True 4,0.102991,0.115739,0.111379,0.113889,0.106043,0.113169,0.113950,0.114904,0.107936
True 5,0.114448,0.117997,0.113327,0.106724,0.112565,0.124692,0.103760,0.103554,0.102932
True 6,0.105938,0.113313,0.099597,0.114815,0.126601,0.113712,0.103086,0.118161,0.104779
True 7,0.107950,0.109691,0.105932,0.112907,0.101233,0.106644,0.120739,0.111796,0.123107
True 8,0.109321,0.121153,0.098109,0.114098,0.106530,0.112680,0.109623,0.111308,0.117177
True 9,0.105745,0.102856,0.122807,0.109764,0.099621,0.097994,0.126249,0.113757,0.121206


In [15]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    poisson_grouped["y_true"],
    poisson_grouped["y_pred"],
    labels=np.arange(1, 10),
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {k}" for k in range(1, 10)],
    columns=[f"Pred {k}" for k in range(1, 10)],
)

cm_df

,Pred 1,Pred 2,Pred 3,Pred 4,Pred 5,Pred 6,Pred 7,Pred 8,Pred 9
True 1,9,3,2,4,11,3,7,6,8
True 2,9,1,7,4,10,8,6,14,8
True 3,2,3,9,2,11,5,5,8,7
True 4,7,3,8,7,4,7,10,9,7
True 5,8,4,5,3,5,10,3,6,7
True 6,6,0,3,7,10,5,3,7,7
True 7,8,1,10,11,3,7,8,6,11
True 8,6,2,2,6,5,5,5,6,12
True 9,3,1,4,6,3,4,13,6,7


In [16]:
from src.poisson_naive_bayes import fit_poisson_nb

model_full = fit_poisson_nb(
    X,
    y,
    uniform_prior=True,
)

print(model_full["rates"].shape)
print(model_full["rates"])

(9, 9)
[[0.67924528 0.52830189 0.88679245 0.58490566 0.62264151 0.45283019
  0.60377358 0.67924528 0.81132075]
 [0.70149254 0.71641791 0.71641791 0.7761194  0.68656716 0.70149254
  0.58208955 0.6119403  0.59701493]
 [0.78846154 0.94230769 0.51923077 0.63461538 0.69230769 0.69230769
  0.78846154 0.69230769 0.82692308]
 [0.5483871  0.72580645 0.61290323 0.61290323 0.74193548 0.70967742
  0.62903226 0.61290323 0.48387097]
 [0.49019608 0.7254902  0.84313725 0.82352941 0.62745098 0.68627451
  0.74509804 0.78431373 0.74509804]
 [0.60416667 0.64583333 0.66666667 0.875      0.79166667 0.77083333
  0.5        0.875      0.6875    ]
 [0.6        0.46153846 0.41538462 0.64615385 0.69230769 0.50769231
  0.72307692 0.78461538 0.6       ]
 [0.79591837 0.59183673 0.69387755 0.69387755 0.51020408 0.57142857
  0.51020408 0.75510204 0.44897959]
 [0.93617021 0.70212766 0.61702128 0.55319149 0.87234043 0.42553191
  0.93617021 0.87234043 0.46808511]]


In [17]:
import pandas as pd

lambda_df = pd.DataFrame(
    model_full["rates"],
    index=[f"Number {k}" for k in range(1, 10)],
    columns=[
        "50-150",
        "150-250",
        "250-350",
        "350-450",
        "450-550",
        "550-650",
        "650-750",
        "750-850",
        "850-950",
    ],
)

lambda_df

,50-150,150-250,250-350,350-450,450-550,550-650,650-750,750-850,850-950
Number 1,0.679245,0.528302,0.886792,0.584906,0.622642,0.452830,0.603774,0.679245,0.811321
Number 2,0.701493,0.716418,0.716418,0.776119,0.686567,0.701493,0.582090,0.611940,0.597015
Number 3,0.788462,0.942308,0.519231,0.634615,0.692308,0.692308,0.788462,0.692308,0.826923
Number 4,0.548387,0.725806,0.612903,0.612903,0.741935,0.709677,0.629032,0.612903,0.483871
Number 5,0.490196,0.725490,0.843137,0.823529,0.627451,0.686275,0.745098,0.784314,0.745098
Number 6,0.604167,0.645833,0.666667,0.875000,0.791667,0.770833,0.500000,0.875000,0.687500
Number 7,0.600000,0.461538,0.415385,0.646154,0.692308,0.507692,0.723077,0.784615,0.600000
Number 8,0.795918,0.591837,0.693878,0.693878,0.510204,0.571429,0.510204,0.755102,0.448980
Number 9,0.936170,0.702128,0.617021,0.553191,0.872340,0.425532,0.936170,0.872340,0.468085


In [18]:
import numpy as np
import pandas as pd

classes = np.arange(1, 10)

fano_matrix = np.full((9, 9), np.nan)
mean_matrix = np.full((9, 9), np.nan)
variance_matrix = np.full((9, 9), np.nan)

for i, k in enumerate(classes):

    X_k = X[y == k]

    # Mean spike count in each 100-ms bin
    means = X_k.mean(axis=0)

    # Sample variance in each bin
    variances = X_k.var(axis=0, ddof=1)

    mean_matrix[i] = means
    variance_matrix[i] = variances

    # Avoid division by zero
    fano = np.divide(
        variances,
        means,
        out=np.full_like(means, np.nan, dtype=float),
        where=means > 0,
    )

    fano_matrix[i] = fano

In [19]:
bin_labels = [
    "50-150",
    "150-250",
    "250-350",
    "350-450",
    "450-550",
    "550-650",
    "650-750",
    "750-850",
    "850-950",
]

fano_df = pd.DataFrame(
    fano_matrix,
    index=[f"Number {k}" for k in classes],
    columns=bin_labels,
)

fano_df

,50-150,150-250,250-350,350-450,450-550,550-650,650-750,750-850,850-950
Number 1,1.459402,0.844780,1.026187,1.212159,1.187646,0.982372,0.786058,1.176282,0.903399
Number 2,0.691812,0.626263,0.837753,0.773893,1.200922,0.994197,0.944833,1.037694,0.865909
Number 3,0.862267,0.932773,1.018882,0.866904,1.333333,0.823529,1.508847,0.993464,0.935249
Number 4,1.296046,1.136976,0.928387,0.714409,1.101924,1.126677,1.158890,0.660915,0.931148
Number 5,0.846400,0.996757,1.108837,1.054286,0.762500,0.844571,0.957895,1.189000,0.904211
Number 6,1.038151,0.822924,0.659574,0.954407,1.126540,0.896492,1.191489,0.905775,0.938104
Number 7,1.135417,1.088542,0.969907,0.697917,0.944444,0.869318,1.145612,1.294118,1.343750
Number 8,1.045940,0.909483,0.912990,1.153186,0.826667,1.020833,0.826667,0.856982,0.933712
Number 9,1.272727,1.109354,0.814093,1.085284,0.728526,0.791304,0.761858,0.927890,0.822134


In [20]:
print("Mean Fano factor:", np.nanmean(fano_matrix))
print("Median Fano factor:", np.nanmedian(fano_matrix))
print("Minimum Fano factor:", np.nanmin(fano_matrix))
print("Maximum Fano factor:", np.nanmax(fano_matrix))

Mean Fano factor: 0.9798610936465963
Median Fano factor: 0.9444444444444439
Minimum Fano factor: 0.6262626262626261
Maximum Fano factor: 1.5088474414155906


In [21]:
valid_fano = fano_matrix[np.isfinite(fano_matrix)]

print(
    "Fraction between 0.8 and 1.2:",
    np.mean((valid_fano >= 0.8) & (valid_fano <= 1.2))
)

print(
    "Fraction between 0.5 and 1.5:",
    np.mean((valid_fano >= 0.5) & (valid_fano <= 1.5))
)

Fraction between 0.8 and 1.2: 0.7407407407407407
Fraction between 0.5 and 1.5: 0.9876543209876543
